In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "openai-community/roberta-large-openai-detector"

print(f"Loading {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

print(f"Model loaded on {device}")


# SCORING FUNCTION

def compute_openai_roberta_score(text, model, tokenizer, device, max_len=512):

    try:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        ai_prob = probs[0][0].item()

        return ai_prob

    except Exception as e:
        return 0.5

# PREPARE FUNCTION

def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER

def evaluate_openai_roberta(texts, labels, model, tokenizer, device):

    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_openai_roberta_score(t, model, tokenizer, device)
        scores.append(s)

    thr = np.median(scores)
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET
print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")


# DEFINE TASKS
tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}


# RUN ALL TASKS
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_openai_roberta(X, y, model, tokenizer, device)
    res[name] = mt

    # Save results
    model_name_clean = MODEL_NAME.split('/')[-1]
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_{model_name_clean}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")


# SUMMARY
summary = pd.DataFrame(res).T
model_name_clean = MODEL_NAME.split('/')[-1]
summary.to_csv(f"{PROJECT}/results/{model_name_clean}_summary.csv")

print("\n" + "="*60)
print(f"SUMMARY")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")


Loading openai-community/roberta-large-openai-detector


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/519 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

AUROC: 0.4865 | Acc: 0.4878 | F1: 0.5902

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [02:11<00:00, 66.37it/s]


AUROC: 0.5969 | Acc: 0.5493 | F1: 0.3990

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [02:09<00:00, 67.35it/s]


AUROC: 0.5379 | Acc: 0.5128 | F1: 0.6102

SUMMARY
         auroc  accuracy      f1
H_vs_M  0.6299    0.5784  0.5782
E_vs_M  0.6387    0.5944  0.5944
H_vs_E  0.4888    0.4897  0.4897
L_vs_M  0.6471    0.5776  0.4368
H_vs_L  0.4865    0.4878  0.5902
G_vs_M  0.5969    0.5493  0.3990
H_vs_G  0.5379    0.5128  0.6102

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/
